In [109]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import h5py
import re

root = "/home/lwu/work/NarrativeGenomics/region_narration/code/pipeline/ARCHS4_LW/refining/data_slash_results/"
filename = root + 'archs4_gene_human_v2.1_normalized_LW20230126.h5'
# filename = "/home/lwu/ARCHS4/Tests/top_10_genesets_per_category/test_cases/test_h5df_data.hdf5"

with h5py.File(filename, 'r') as f:
    logOE_matrix = f['data/logOE_matrix'][:]
    gene_names = f['meta/genes/all_gene_names'][:]
    category_names = f['meta/biosamples/category'][:]

category_names = [x.decode('UTF-8') for x in category_names]
archs_gene_names = [x.decode('UTF-8') for x in gene_names]


category_list  = {
    "Heart", "Atrium", "Cardiac Muscle Fiber", "Pericardium", "Valve", "Ventricle","HUVEC", 
    "Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
    "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF",
    "HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
    "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
    "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
    "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus","Granulocyte", "Neutrophil", "Blymphocyte", 
    "Plasma Cell", "Plasmacytoid Dendritic Cell", 
    "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
    "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
    "Thymocyte", "Spleen",
    "Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
    "A431", "HACAT", "HNSCC","Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
    "Vascular Smooth Muscle", "LHCNM2","Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
    "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
    "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron","Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
    "Trachea","Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
    "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
    "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"
}
category_dict  = {
    "Cardiovascular System" : ["Heart", "Atrium", "Cardiac Muscle Fiber", 
                               "Pericardium", "Valve", "Ventricle",
                               "HUVEC"], 
    
    "Connective Tissue" : ["Adipose", "Adipocyte", "Myofibroblast", "Osteoblast", "Stromal Cell", 
                           "MG63", "K562", "Bone Marrow", "Chondrocyte", "HT1080", "NHDF"],

    "Digestive System": ["HCT116", "HT29", "RKO", "Colon", "Colonic Muscosa", "Intestinal Epithelial Cell", 
                         "Small Intestine", "Liver", "Hepatic Stellate Cell", "Hepatocyte", "Primary Liver Cell",
                         "HEPG2", "Alpha Cell", "Beta Cell", "Pancreatic Islet", 
                         "GSE81547", "PANC1", "Gastric Epithelial Cell", "Esophagus"],
    
    "Immune System" : ["Granulocyte", "Neutrophil", "Blymphocyte", "Plasma Cell", "Plasmacytoid Dendritic Cell", 
                       "Tlymphocyte", "GSM2679484", "JURKAT", "MT4", "NALM6", "REH", "RPMI8226", "Alveolar", 
                       "Dendritic", "Kupffer", "Macrophage", "Macroglia",  "THP1", "U937", "JEG3", "Thymus", 
                       "Thymocyte", "Spleen"],
    
    "Integumentary System" : ["Skin", "Basal Cell", "Fibroblast", "Hair Follicle", "Keratinocyte", "Melanocyte",
                              "A431", "HACAT", "HNSCC"], 
    
    "Muscular System" : ["Skeletal Muscle", "Myoblast", "Airway Smooth Muscle", "Respiratory Smooth Muscle", 
                         "Vascular Smooth Muscle", "LHCNM2"], 
    
    "Nervous System" : ["Brain", "DAOY", "IMR32", "SKNSH", "Astrocyte", "Cerebellum", "Cerebral Cortex", 
                        "Hypothalamus", "Medula", "Midbrain", "Neuron", "Oligodendrocyte", "Pons", "Spinal Cord", 
                        "Thalamus", "Retina", "Motor Neuron", "Sensory Neuron"], 
    
    "Respiratory System" : ["Lung", "Lung Epithelial", "A549", "H1299", "IMR90", "MRC5", "NCIH460", "NHBE", 
                            "Trachea"], 
    
    "Urogenital System" : ["Breast", "Mammary Gland", "MCF7", "MCF10", "MDAMB231", "MDAMB468", "SKBR3", "T47D", 
                           "HELA", "Kidney", "Podocyte", "Renal Cortex", "293F", "FLPIN TREX 293", "HEK293", "C42", 
                           "DU145", "LNCAP", "PC3", "Ovary", "Granulosa", "Oocyte", "ES2", "SKOV3", "Testis"]
}

# Convert category_dict value list elements to lowercase
for category, value_list in category_dict.items():
    for i in range(len(value_list)):
        value_list[i] = value_list[i].lower()

In [110]:
folder = "/home/lwu/ARCHS4/Tests/Check If Biosample Categorys Align With Genesets/"
filename = folder + "h.c1.c2.all.v7.5.1.symbols_removedMT.gmt"

# filename = "/home/lwu/ARCHS4/Tests/top_10_genesets_per_category/test_cases/test_geneset.gmt"

---
# Functions

In [111]:
# Get category-sample indexes
all_cat_idxs = {}
for system in category_dict:
    print(system)
    temp_dict = {}
    for element in category_dict[system]:
        indexes = []
        for i, name in enumerate(category_names):
            if element.lower() in name.lower():
                indexes.append(i)
        temp_dict[element] = indexes
    all_cat_idxs[system] = temp_dict

Cardiovascular System
Connective Tissue
Digestive System
Immune System
Integumentary System
Muscular System
Nervous System
Respiratory System
Urogenital System


In [112]:
def get_archs4_gene_data(gene_names):
    gene_indexes = []
    new_gene_names = []
    for gene in gene_names:
        try:
            index = archs_gene_names.index(gene)
            gene_indexes.append(index)
            new_gene_names.append(gene)
        except ValueError:
            pass
    return new_gene_names, gene_indexes

gene_names, gene_indexes = get_archs4_gene_data(gene_names)

In [113]:
def get_category_biosampleMatrix(extracted_matrix):
    
    system_biosampleMatrix = {}

    for system, tier1_list in all_cat_idxs.items():
        for tier1_cat in tier1_list:
            idx_list = tier1_list[tier1_cat]

            matrix = extracted_matrix[:,idx_list].tolist()
            avg = np.mean(matrix, axis=0)
            system_biosampleMatrix[tier1_cat] = avg
            
    return system_biosampleMatrix

In [ ]:
set_geneset_name = "HALLMARK_TNFA_SIGNALING_VIA_NFKB"

all_genesets_matrix_mean = {}
all_genesets_matrix_oe = []
geneset_names = []
gene_names = []


with open(filename, 'r', ) as file:
    
    for line in file:
        #print(line)
        line = line.strip().split('\t')
        geneset_names.append(line[0])
        #print(line[0])
        
#         if line[0] == set_geneset_name: 
            
        gene_names = line[2:]
        #print(geneset_names, ":", gene_names)

        gene_names, gene_indexes = get_archs4_gene_data(gene_names)
#             print(gene_names, "\n", gene_indexes)

        extracted_matrix = logOE_matrix[gene_indexes, :] 
        #print(extracted_matrix)
        #print("Extracted Matrix Shape:", extracted_matrix.shape)

        category_biosampleMatrix = get_category_biosampleMatrix(extracted_matrix)
#             print("----- Result ------")
#             print(category_biosampleMatrix)

        all_genesets_matrix_mean[line[0]] = category_biosampleMatrix
